# Figure: O2 Vapor Mass Balance

Per tool, a log-log scatter of *reported* vs. *by-difference* vapor O2 mole fraction.  Each point is one pressure step of one sample.  The 1:1 line shows perfect agreement; departures indicate internal mass-balance drift in the tool during degassing.

Only tools that report vapor mole fractions for every species and an independently-computed by-difference value show up. Sulfur_X and all VESIcal sub-models don't expose the inputs, so they're skipped.

MAGEC's by-difference values come out negative, so we plot their absolute value and flag it in the panel.


In [ ]:
from pathlib import Path

results_directory = Path().resolve().parent / "Model_Outputs"
SAVE_FIG = True

## Import data and styling

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import pandas as pd

from helpers.plot_styles import (
    AXIS_LABEL_FONTSIZE,
    TICK_FONTSIZE,
    TOOL_COLORS_HEX,
    ANNOTATION_FONTSIZE,
    LEGEND_STYLE,
    SAVE_DPI,
    SAMPLE_DISPLAY_NAMES,
    SYSTEM_COLORS
)
from helpers.degassing_data import load_all_systems, load_system

# --- USER INPUTS --- #
SAMPLES = ["MORB", "Kilauea", "Fuego", "Fogo"]
TOOLS  = ["DCompress (IM)", "EVo", "MAGEC", "VolFe",]
ABS_Y_MODELS  = {"MAGEC"}


systems = load_all_systems(SAMPLES, TOOLS, results_dir=results_directory)


## Combined figure: O2 mass balance + representative closure

Stacks the per-tool O2 mass balance scatter (top row) with the vapor mole-fraction closure check (bottom row) into a single 8-panel figure. The closure row shows Kilauea as a representative system.

### Compute closure error for each tool

In [ ]:
# --- Load data and compute closure error per (tool, sample) -----------------

# Columns that look like _v_mf but are not species mole fractions
_SKIP_COLS = {"CS_v_mf", "XO2_BYDIFF_v_mf", "SUM_v_mf", "CS_v_mf_old"}

def _mf_cols(df: pd.DataFrame) -> list[str]:
    """Return the true vapor species mole-fraction columns in *df*."""
    return [c for c in df.columns if c.endswith("_v_mf") and c not in _SKIP_COLS]

# Load closure data for bottom row
data: dict[str, dict[str, tuple | None]] = {}
active_tools: list[str] = []

for tool in TOOLS:
    tool_data: dict[str, tuple | None] = {}
    has_any = False
    for sample in SAMPLES:
        dfs = load_system(sample, [tool], results_dir=results_directory)
        df = dfs.get(tool)
        if df is None:
            tool_data[sample] = None
            continue
        cols = _mf_cols(df)
        if not cols or "P_bars" not in df.columns:
            tool_data[sample] = None
            continue
        # Keep rows where P_bars is valid; fill NaN species as 0
        # (some tools leave a species column blank when it is absent)
        sub = df[["P_bars"] + cols].dropna(subset=["P_bars"])
        if sub.empty:
            tool_data[sample] = None
            continue
        mf_vals = sub[cols].fillna(0.0)
        sum_x = mf_vals.sum(axis=1)
        closure = sum_x - 1.0
        tool_data[sample] = (sub["P_bars"].values, closure.values)
        has_any = True
    data[tool] = tool_data
    if has_any:
        active_tools.append(tool)

### Build the Figure

In [ ]:
fig, axes = plt.subplots(2, len(TOOLS), figsize=(5 * len(TOOLS), 4.5 * 2), squeeze=False)
sample_colors = list(SYSTEM_COLORS.values())
PANEL_LABELS = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"

# Top row: scatter, all systems overlaid per tool panel
for col, model in enumerate(TOOLS):
    ax = axes[0, col]
    for enum, sample in enumerate(SAMPLES):
        df = systems.get(sample, {}).get(model)
        if df is None or "O2_v_mf" not in df.columns or "XO2_BYDIFF_v_mf" not in df.columns:
            continue
        y = df["XO2_BYDIFF_v_mf"]
        if model in ABS_Y_MODELS:
            y = y.abs()
        ax.scatter(df["O2_v_mf"], y,
                   s=15, color=sample_colors[enum % len(sample_colors)],
                   edgecolors="k", linewidths=0.4, marker="D", alpha=0.9,
                   label=sample if col == 0 else None)

    xlims, ylims = ax.get_xlim(), ax.get_ylim()
    ref_min = min(xlims[0], ylims[0])
    ref_max = max(xlims[1], ylims[1])
    ax.plot([ref_min, ref_max], [ref_min, ref_max], "k-", linewidth=1)

    ax.set_xlabel(r"Reported $X_{O_2}^{vapor}$ (molar frac)")
    ylabel = r"By Difference $X_{O_2}^{vapor}$ (molar frac)"
    if model in ABS_Y_MODELS:
        ylabel = r"|By Difference $X_{O_2}^{vapor}$| (molar frac)"
        ax.text(0.05, 0.95,
                "y-axis: absolute value\n(raw values are negative)",
                transform=ax.transAxes, ha="left", va="top",
                fontsize=ANNOTATION_FONTSIZE, style="italic", color="dimgray")
    ax.set_ylabel(ylabel)
    ax.text(0.95, 0.05, f"{PANEL_LABELS[col]}) {model}",
            transform=ax.transAxes, ha="right", va="bottom",
            fontsize=ANNOTATION_FONTSIZE, fontweight="bold")
    ax.set_xscale("log")
    ax.set_yscale("log")

axes[0, 0].legend(loc="upper left", **LEGEND_STYLE)

# Bottom row: closure trace for Kilauea only, one tool per panel
LINTHRESH = 1e-17
rep_label = "Kilauea"

# Shared y-limit: symmetric about 0, sized to the largest |closure| across the row
y_max = 0.0
for tool in TOOLS:
    pair = data.get(tool, {}).get("Kilauea")
    if pair is not None:
        _, err = pair
        y_max = max(y_max, float(abs(err).max()))

for col, tool in enumerate(TOOLS):
    ax = axes[1, col]
    pair = data.get(tool, {}).get("Kilauea")

    if pair is not None:
        P, err = pair
        color = TOOL_COLORS_HEX.get(tool, "gray")
        ax.plot(P, err, color=color, linewidth=1.2, alpha=0.85)
        ax.axhline(0, color="black", linewidth=0.7, linestyle="--", alpha=0.6)

        ax.set_yscale("symlog", linthresh=LINTHRESH)
        if y_max > 0:
            ax.set_ylim(-y_max, y_max)
        ax.yaxis.set_major_locator(
            ticker.SymmetricalLogLocator(base=10, linthresh=LINTHRESH, subs=[1.0])
        )
        ax.yaxis.get_major_locator().set_params(numticks=6)
        ax.yaxis.set_major_formatter(ticker.LogFormatterSciNotation(labelOnlyBase=True))
    else:
        ax.text(
            0.5, 0.5, "no data",
            transform=ax.transAxes,
            ha="center", va="center",
            fontsize=TICK_FONTSIZE,
            color="gray",
            style="italic",
        )
        ax.set_xticks([])
        ax.set_yticks([])

    ax.set_xlabel("Pressure (bars)")
    if col == 0:
        ax.set_ylabel(rf"$\Sigma x_i - 1$  ({rep_label})")
    ax.text(0.95, 0.05, f"{PANEL_LABELS[len(TOOLS) + col]}) {tool}",
            transform=ax.transAxes, ha="right", va="bottom",
            fontsize=ANNOTATION_FONTSIZE, fontweight="bold")

fig.tight_layout()

if SAVE_FIG:
    fig.savefig("figures/Fig_o2_mass_balance_combined.png", dpi=SAVE_DPI, bbox_inches="tight")